In [3]:
import rasterio
import os
import numpy as np
import matplotlib.pyplot as plt

import json

In [4]:
sentinel_root_dir = '../data/raw/sentinel_2021/'
sentinel_dirs =[os.path.join(sentinel_root_dir, x) for x in os.listdir(sentinel_root_dir)]
sentinel_dirs

['../data/raw/sentinel_2021/.ipynb_checkpoints',
 '../data/raw/sentinel_2021/S2B_MSIL2A_20210513T073609_R092_T36KUU_20210514T122203',
 '../data/raw/sentinel_2021/S2B_MSIL2A_20210513T073609_R092_T36KVU_20210606T053436',
 '../data/raw/sentinel_2021/S2B_MSIL2A_20210513T073609_R092_T36JUT_20210514T161910']

In [5]:
channels = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B09', 'B11', 'B12']
image_stats_by_tile = {}
for channel in channels:
    image_stats_by_tile[channel] = []
image_stats_by_tile['pixel_count'] = []

for sentinel_dir in sentinel_dirs:
    for fn in os.listdir(sentinel_dir):
        if not fn.endswith('.tif'): continue
        channel = fn.split('_')[2].split('.')[0]
        if channel.startswith('B'):
            with rasterio.open(os.path.join(sentinel_dir, fn)) as f:

                data = f.read()

                this_dict = {
                    # 'mean': np.mean(data),
                    # 'std': np.std(data),
                    # 'max': np.max(data), 
                    # 'num_pix': f.shape[0] * f.shape[1],
                    'vals': data.ravel()
                }
                
                image_stats_by_tile[channel].append(this_dict)

In [6]:
image_stats_by_channel = {}
for channel in channels:
    all_pix = []
    for x in image_stats_by_tile[channel]:
        all_pix.append(x['vals'])
    image_stats_by_channel[channel] = {'mean': np.mean(all_pix),
                                       'std': np.std(all_pix)
                                       }

In [16]:
(np.hstack(all_pix) == 0).sum()

14

In [8]:
image_stats_by_channel

{'B01': {'mean': 311.5623321090507, 'std': 98.76506187808876},
 'B02': {'mean': 441.49430911532033, 'std': 139.81895559350932},
 'B03': {'mean': 628.1743687562417, 'std': 171.20214487660158},
 'B04': {'mean': 728.8783480341251, 'std': 295.3603888999678},
 'B05': {'mean': 1120.0048912910042, 'std': 286.86695796751707},
 'B06': {'mean': 1850.9966619332163, 'std': 299.87618367178163},
 'B07': {'mean': 2148.3984699011176, 'std': 353.0374101607255},
 'B08': {'mean': 2331.6502644068864, 'std': 396.46716984620616},
 'B8A': {'mean': 2433.2913813269806, 'std': 387.27503172230485},
 'B09': {'mean': 2458.9950342898665, 'std': 367.3343672167844},
 'B11': {'mean': 2294.4210610228015, 'std': 535.0164366856719},
 'B12': {'mean': 1472.2561474469783, 'std': 460.9961636512685}}

In [9]:
data_stats_dir = "../data/int/data_stats"
if not os.path.exists(data_stats_dir): os.mkdir(data_stats_dir)
save_fp = os.path.join(data_stats_dir, "S2_stats_by_channel.json")
json.dump(image_stats_by_channel, open(save_fp, 'w' ) )


In [ ]:
sentinel_layer_codes = {"b": "B02",
                        "g": "B03",
                        "r": "B04",
                        "nir":"B08",
                        "vis":"TCI",
                       }



In [17]:
rgbnir_codes = ["B04", "B03","B02","B08"]
sentinel_layer_means = [image_stats_by_channel[channel]['mean'] for channel in rgbnir_codes]

In [18]:
sentinel_layer_means

[728.8783480341251, 628.1743687562417, 441.49430911532033, 2331.6502644068864]